In [16]:
from PIL import Image
import numpy as np

# 读取图像
image_path = '0001_rgb.jpg'
image = Image.open(image_path)
image_array = np.array(image)

# 获取图像尺寸
width, height = image.size

# 检查图像模式
if image.mode == 'RGB':
    black_color = [0, 0, 0]
    print('rgb')
elif image.mode == 'RGBA':
    black_color = [0, 0, 0, 255]
    print('rgba')
else:
    raise ValueError("Unsupported image mode. Only RGB and RGBA are supported.")

# 假设黑线宽度大约为5像素，可以根据实际情况调整
line_width_threshold = 5

# 寻找黑线所在的位置
black_line_positions = []
x = 0
while x < width:
    column = image_array[:, x]  # 取出每一列
    # 检查整列是否为纯黑色
    if np.all(column == black_color):
        black_line_positions.append(x)
        # 跳过黑线的宽度，避免重复检测
        x += line_width_threshold
    else:
        x += 1

# 计算比例
black_line_ratios = [str(pos / width * 100) + '%' for pos in black_line_positions]

ratio_of_angle = [int(pos / width * 360) for pos in black_line_positions]

# 输出比例
print("黑線所在的水平比例:", black_line_ratios)
print(f'ratio of angle: {ratio_of_angle}')


rgb
黑線所在的水平比例: []
ratio of angle: []


In [61]:
from PIL import Image
import numpy as np


def getRatio(image_path):
    print(image_path)
    image = Image.open(image_path)
    image_array = np.array(image)

    # 获取图像尺寸
    width, height = image.size

    # 检查图像模式
    if image.mode == 'RGB':
        black_color = [0, 0, 0]
        print('rgb')
    elif image.mode == 'RGBA':
        black_color = [0, 0, 0, 255]
        print('rgba')
    else:
        raise ValueError("Unsupported image mode. Only RGB and RGBA are supported.")

    # 假设黑线宽度大约为5像素，可以根据实际情况调整
    line_width_threshold = 5

    # 容差值，用于判断像素接近黑色的程度
    tolerance = 30

    # 寻找黑线所在的位置
    black_line_positions = []
    x = 0
    while x < width:
        column = image_array[:, x]  # 取出每一列
        # 检查整列是否接近黑色
        if np.all(np.linalg.norm(column - black_color, axis=-1) < tolerance):
            black_line_positions.append(x)
            # 跳过黑线的宽度，避免重复检测
            x += line_width_threshold
        else:
            x += 1

    # 计算比例
    black_line_ratios = [str(pos / width * 100) + ' %' for pos in black_line_positions]

    ratio_of_angle = [int(pos / width * 360) for pos in black_line_positions]

    # 输出比例
    print("黑线所在的水平比例:", black_line_ratios)
    print(f'ratio of angle: {ratio_of_angle}')

image_path = '0011_rgb.jpg'
getRatio(image_path=image_path)

0011_rgb.jpg
rgb
黑线所在的水平比例: ['11.42578125 %', '41.69921875 %', '73.14453125 %', '81.25 %', '91.796875 %']
ratio of angle: [41, 150, 263, 292, 330]


# cal metrics

In [74]:
import os
import re

# 定义要处理的目录和保存结果的文件名
start = 1
end = 20
directory = '../record/matterport3d/baseline_unifuse/depthAnythingV2_metric_raw/result/'  # 指定文件夹路径
output_file = f'../record/matterport3d/baseline_unifuse/depthAnythingV2_metric_raw/average_results_{start}_{end}.txt'

# 用于存储各个值的总和
sum_mse_result = 0
sum_mae_result = 0
sum_mre_result = 0
sum_mselog_result = 0
sum_delta1_result = 0
sum_delta2_result = 0
sum_delta3_result = 0

file_count = 0

# 正则表达式匹配 'result' 值
result_pattern = re.compile(r'(mse|mae|mre|mselog|delta1|delta2|delta3)_result:\s([\d\.]+)')

filenames = [f'{n:04}.aligned.txt' for n in range(start, end+1)] 

# 遍历文件夹中的所有 .txt 文件
# for filename in os.listdir(directory): 
for filename in filenames: 
    if filename.endswith(".txt"):
        print(filename)
        file_count += 1
        with open(os.path.join(directory, filename), 'r') as file:
            content = file.read()
            
            # 找到所有匹配的 'result' 值并累加
            for match in result_pattern.findall(content):
                metric, value = match[0], float(match[1])
                if metric == 'mse':
                    sum_mse_result += value
                elif metric == 'mae':
                    sum_mae_result += value
                elif metric == 'mre':
                    sum_mre_result += value
                elif metric == 'mselog':
                    sum_mselog_result += value
                elif metric == 'delta1':
                    sum_delta1_result += value
                elif metric == 'delta2':
                    sum_delta2_result += value
                elif metric == 'delta3':
                    sum_delta3_result += value

# 计算平均值
average_mse_result = sum_mse_result / file_count
average_mae_result = sum_mae_result / file_count
average_mre_result = sum_mre_result / file_count
average_mselog_result = sum_mselog_result / file_count
average_delta1_result = sum_delta1_result / file_count
average_delta2_result = sum_delta2_result / file_count
average_delta3_result = sum_delta3_result / file_count

# 将平均值写入新文件
with open(output_file, 'w') as f_out:
    f_out.write(f"mse_result: {average_mse_result:.6f}\n")
    f_out.write(f"mae_result: {average_mae_result:.6f}\n")
    f_out.write(f"mre_result: {average_mre_result:.6f}\n")
    f_out.write(f"mselog_result: {average_mselog_result:.6f}\n")
    f_out.write(f"delta1_result: {average_delta1_result:.6f}\n")
    f_out.write(f"delta2_result: {average_delta2_result:.6f}\n")
    f_out.write(f"delta3_result: {average_delta3_result:.6f}\n")

print(f"处理完成，结果已保存到 {output_file}")
print(f'file count: {file_count}')


0001.aligned.txt
0002.aligned.txt
0003.aligned.txt
0004.aligned.txt
0005.aligned.txt
0006.aligned.txt
0007.aligned.txt
0008.aligned.txt
0009.aligned.txt
0010.aligned.txt
0011.aligned.txt
0012.aligned.txt
0013.aligned.txt
0014.aligned.txt
0015.aligned.txt
0016.aligned.txt
0017.aligned.txt
0018.aligned.txt
0019.aligned.txt
0020.aligned.txt
处理完成，结果已保存到 ../record/matterport3d/baseline_unifuse/depthAnythingV2_metric_raw/average_results_1_20.txt
file count: 20


# cal metrics diff

In [75]:
# 定义读取文件并解析内容的函数
def parse_file(file_path):
    keys = []
    values = []
    with open(file_path, 'r') as file:
        for line in file:
            key, value = line.strip().split(': ')
            keys.append(key)
            values.append(float(value))
    return keys, values

# 定义计算百分比差异的函数
def calculate_percentage_difference(results1, results2):
    percentage_differences = []
    for value1, value2 in zip(results1, results2):
        percentage_difference = ((value1 - value2) / value2) * 100
        percentage_differences.append(percentage_difference)
    return percentage_differences

# 读取两个文件
baselines = ['baseline_unifuse']
perspectives = ['depthAnything_metric_raw', 'depthAnythingV2_metric_raw']
types = ['_test1', '_test2']
start = 11
end = 20

for baseline in baselines:
    for perspective in perspectives:
        for type in types:
            if baseline == 'baseline_hohonet':
                file2_types = ['leres', f'leres{type}', f'{perspective}_scaled']
            elif perspective == 'leres':
                file2_types = ['leres']
            else:
                file2_types = [f'{perspective}']
            for file2_type in file2_types:
        
                file1_path = f'../record/matterport3d/{baseline}/{perspective}{type}/average_results_{start}_{end}.txt'
                file2_path = f'../record/matterport3d/{baseline}/{file2_type}/average_results_{start}_{end}.txt'
                
                keys1, results1 = parse_file(file1_path)
                _, results2 = parse_file(file2_path)

                # 计算百分比差异
                percentage_differences = calculate_percentage_difference(results1, results2)

                # 打印结果
                print(f'../record/matterport3d/{baseline}/{perspective}{type}  VS  ../record/matterport3d/{baseline}/{file2_type}')

                output_file_path = f'{start}_{end} : ../record/matterport3d/{baseline}/{perspective}{type}/vs_{file2_type}_{start}_{end}.txt'
                with open(output_file_path, 'w') as output_file:
                    for key, value in zip(keys1, percentage_differences):
                        output_file.write(f"{key}: {value:.2f}%\n")


../record/matterport3d/baseline_unifuse/depthAnything_metric_raw_test1  VS  ../record/matterport3d/baseline_unifuse/depthAnything_metric_raw
../record/matterport3d/baseline_unifuse/depthAnything_metric_raw_test2  VS  ../record/matterport3d/baseline_unifuse/depthAnything_metric_raw
../record/matterport3d/baseline_unifuse/depthAnythingV2_metric_raw_test1  VS  ../record/matterport3d/baseline_unifuse/depthAnythingV2_metric_raw
../record/matterport3d/baseline_unifuse/depthAnythingV2_metric_raw_test2  VS  ../record/matterport3d/baseline_unifuse/depthAnythingV2_metric_raw
